In [ ]:
import numpy as np
from lidar_object_tracking.config import PROCESSED_DATA_DIR
from lidar_object_tracking.dataset import json_to_numpy
from lidar_object_tracking.features import dbscan_clustering_labels, pull_clusters_fill_frames, find_cluster_centers

In [ ]:
dataset = json_to_numpy(PROCESSED_DATA_DIR,'004')
labels = dbscan_clustering_labels(dataset = dataset)
processed_labeled_dataset = pull_clusters_fill_frames(dataset,labels, label = 6)

In [ ]:
min_key = np.min(list(processed_labeled_dataset.keys()))
max_key = np.max(list(processed_labeled_dataset.keys()))
frame_rate = 0.1

cluster_centers = find_cluster_centers(processed_labeled_dataset)

# Reduce to an xy coordinate system since objects on the street generally don't change vertical position.
x_centers = np.array([cluster_centers[key][0] for key in cluster_centers.keys()])
y_centers = np.array([cluster_centers[key][1] for key in cluster_centers.keys()])

xy_centers = np.array([x_centers,y_centers])

In [ ]:
# rows: x, y, dx, dy
state_transition = np.array([[1, 0, frame_rate , 0],
                             [0, 1, 0, frame_rate],
                             [0, 0, 1, 0],
                             [ 0, 0 , 0 ,1]])

In [ ]:
from pykalman import KalmanFilter

kf = KalmanFilter(transition_matrices = state_transition,
                 observation_matrices= np.array([[1,0,0,0],
                                                [0,1,0,0]]))
measurements = xy_centers.T

kf.em(measurements, n_iter=5)
(filtered_state_means, filtered_state_covariances) = kf.filter(measurements)
(smoothed_state_means, smoothed_state_covariances) = kf.smooth(measurements)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(x = smoothed_state_means[:,0],
               y = smoothed_state_means[:,1]
                )
)
fig.update_yaxes(
    scaleanchor = 'x'
)